In [1]:
!pip install huggingface_hub==0.23.0

In [2]:
# ============================================================
# CELL 1 — Imports
# ============================================================
import os
import torch
import torchaudio
import pandas as pd
from speechbrain.inference.speaker import EncoderClassifier

print("✅ All imports done")

/Users/abey/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
SpeechBrain could not find any working torchaudio backend. Audio files may fail to load. Follow this link for instructions and troubleshooting: https://speechbrain.readthedocs.io/en/latest/audioloading.html


✅ All imports done


In [3]:
# ============================================================
# CELL 2 — Paths and threshold
# ============================================================
BASE_DIR        = "/Users/abey/Documents/speaker_similarity"  # ← change this
REFERENCE_DIR   = os.path.join(BASE_DIR, "reference")
ENROLLMENT_FILE = os.path.join(BASE_DIR, "enrollment", "speaker.wav")
MODELS_DIR      = os.path.join(BASE_DIR, "models")

# threshold — calibrate with real cross-lingual data later
SPEAKER_SIM_THRESHOLD = 0.75

print(f"BASE_DIR        : {BASE_DIR}")
print(f"REFERENCE_DIR   : {REFERENCE_DIR}")
print(f"ENROLLMENT_FILE : {ENROLLMENT_FILE}")
print(f"MODELS_DIR      : {MODELS_DIR}")
print(f"Threshold       : {SPEAKER_SIM_THRESHOLD}")
print("✅ Paths and threshold set")

BASE_DIR        : /Users/abey/Documents/speaker_similarity
REFERENCE_DIR   : /Users/abey/Documents/speaker_similarity/reference
ENROLLMENT_FILE : /Users/abey/Documents/speaker_similarity/enrollment/speaker.wav
MODELS_DIR      : /Users/abey/Documents/speaker_similarity/models
Threshold       : 0.75
✅ Paths and threshold set


In [4]:
# ============================================================
# CELL 3 — Load ECAPA-TDNN model
# ============================================================
classifier = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    run_opts={"device": "cpu"}
)

print("✅ ECAPA-TDNN loaded")

SpeechBrain could not find any working torchaudio backend. Audio files may fail to load. Follow this link for instructions and troubleshooting: https://speechbrain.readthedocs.io/en/latest/audioloading.html
/Users/abey/miniconda3/lib/python3.13/site-packages/speechbrain/utils/autocast.py:188: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  wrapped_fwd = torch.cuda.amp.custom_fwd(fwd, cast_inputs=cast_inputs)


✅ ECAPA-TDNN loaded


In [5]:
# ============================================================
# CELL 4 — Scoring functions
# ============================================================
def get_embedding(audio_path):
    signal, sr = torchaudio.load(audio_path)

    if signal.shape[-1] < 160:
        raise ValueError(f"Audio too short: {audio_path}")

    
    if signal.shape[0] > 1:
        signal = signal.mean(dim=0, keepdim=True)
    
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(
            orig_freq=sr,
            new_freq=16000
        )
        signal = resampler(signal)
    
    with torch.no_grad():
        embedding = classifier.encode_batch(signal)
    
    return embedding.squeeze()

def cosine_similarity(emb1, emb2):
    emb1_norm = emb1 / torch.norm(emb1)
    emb2_norm = emb2 / torch.norm(emb2)
    return round(float(torch.dot(emb1_norm, emb2_norm)), 4)

def compute_speaker_sim(reference_path, tts_path):
    ref_emb = get_embedding(reference_path)
    tts_emb = get_embedding(tts_path)
    return cosine_similarity(ref_emb, tts_emb)

print("✅ compute_speaker_sim defined")

✅ compute_speaker_sim defined


In [6]:
# ============================================================
# CELL 5 — Sanity check
# ============================================================
test_path = "/Users/abey/Documents/suits_base_sample.wav"

same_score = compute_speaker_sim(test_path, test_path)
print(f"Same file similarity : {same_score}")
print(f"Expected             : close to 1.0")

Same file similarity : 1.0
Expected             : close to 1.0


In [7]:
# ============================================================
# CELL 6 — Startup validation
# ============================================================
if not os.path.exists(MODELS_DIR):
    raise FileNotFoundError(f"Models folder not found: {MODELS_DIR}")

enrollment_available = os.path.exists(ENROLLMENT_FILE)
if enrollment_available:
    print(f"✅ Enrollment file found")
else:
    print(f"⚠️  No enrollment file — segments without utterance ref will be skipped")

reference_available = os.path.exists(REFERENCE_DIR)
if reference_available:
    ref_files = sorted([f for f in os.listdir(REFERENCE_DIR) if f.endswith(".wav")])
    print(f"✅ Reference folder found: {len(ref_files)} utterance files")
else:
    print(f"⚠️  No reference folder — using enrollment for all segments")

model_folders = sorted([
    d for d in os.listdir(MODELS_DIR)
    if os.path.isdir(os.path.join(MODELS_DIR, d))
])
if not model_folders:
    raise ValueError(f"No model folders found in {MODELS_DIR}")
print(f"✅ Models found: {model_folders}")

model_samples = {}
for model in model_folders:
    model_path = os.path.join(MODELS_DIR, model)
    wav_files  = sorted([
        f for f in os.listdir(model_path)
        if f.endswith(".wav")
    ])
    model_samples[model] = wav_files
    print(f"   {model}: {len(wav_files)} samples")

reference_filenames = set(model_samples[model_folders[0]])
for model in model_folders[1:]:
    current_filenames = set(model_samples[model])
    if current_filenames != reference_filenames:
        missing = reference_filenames - current_filenames
        extra   = current_filenames - reference_filenames
        raise ValueError(
            f"Model '{model}' has mismatched filenames.\n"
            f"  Missing : {missing}\n"
            f"  Extra   : {extra}"
        )
print("✅ All models have identical filenames")

sample_names = model_samples[model_folders[0]]
total        = len(model_folders) * len(sample_names)
print(f"\nReady: {len(model_folders)} models × {len(sample_names)} samples = {total} evaluations")
print(f"Reference  : {'available' if reference_available else 'not available'}")
print(f"Enrollment : {'available' if enrollment_available else 'not available'}")

✅ Enrollment file found
✅ Reference folder found: 2 utterance files
✅ Models found: ['m1']
   m1: 2 samples
✅ All models have identical filenames

Ready: 1 models × 2 samples = 2 evaluations
Reference  : available
Enrollment : available


In [8]:
# ============================================================
# CELL 7 — Main evaluation loop
# ============================================================
results = []

for model in model_folders:
    print(f"\n{'='*50}")
    print(f"Model: {model}")
    print(f"{'='*50}")

    for wav_file in model_samples[model]:
        sample_name = os.path.splitext(wav_file)[0]
        tts_path    = os.path.join(MODELS_DIR, model, wav_file)

        print(f"\n  Sample : {sample_name}")

        utterance_path = os.path.join(REFERENCE_DIR, wav_file) if reference_available else None

        if utterance_path and os.path.exists(utterance_path):
            reference_path = utterance_path
            ref_type       = "UTTERANCE_REF"
        elif enrollment_available:
            reference_path = ENROLLMENT_FILE
            ref_type       = "ENROLLMENT_REF"
        else:
            print(f"  ⚠️  No reference available — skipping")
            results.append({
                "Model"   : model,
                "Sample"  : sample_name,
                "Score"   : None,
                "Pass"    : "⚠️ SKIP",
                "Ref Type": "NO_REF",
            })
            continue

        score  = compute_speaker_sim(reference_path, tts_path)
        passed = score >= SPEAKER_SIM_THRESHOLD

        print(f"  Score  : {score} | {'✅ PASS' if passed else '❌ FAIL'} | {ref_type}")

        results.append({
            "Model"   : model,
            "Sample"  : sample_name,
            "Score"   : score,
            "Pass"    : "✅" if passed else "❌",
            "Ref Type": ref_type,
        })

print("\n\nAll evaluations complete.")


Model: m1

  Sample : YASH 2
  Score  : 0.965 | ✅ PASS | UTTERANCE_REF

  Sample : YASH_01
  Score  : 0.965 | ✅ PASS | UTTERANCE_REF


All evaluations complete.


In [9]:
# ============================================================
# CELL 8 — Results and model comparison
# ============================================================
df = pd.DataFrame(results)

print("\n========== FULL PER-SEGMENT RESULTS ==========")
print(df[["Model", "Sample", "Score", "Pass", "Ref Type"]].to_string(index=False))

print("\n========== MODEL COMPARISON SUMMARY ==========")
summary_rows = []

for model in model_folders:
    model_df         = df[df["Model"] == model]
    valid_df         = model_df[model_df["Score"].notna()]
    scores           = valid_df["Score"]
    pass_count       = (model_df["Pass"] == "✅").sum()
    total            = len(model_df)
    enrollment_count = (model_df["Ref Type"] == "ENROLLMENT_REF").sum()

    summary_rows.append({
        "Model"           : model,
        "Segments"        : total,
        "Mean Score"      : round(scores.mean(), 4) if len(scores) > 0 else None,
        "Median Score"    : round(scores.median(), 4) if len(scores) > 0 else None,
        "Min Score"       : round(scores.min(), 4) if len(scores) > 0 else None,
        "Pass Rate"       : f"{pass_count}/{total}",
        "Enrollment Rate" : f"{enrollment_count}/{total}",
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

print("\n========== MODEL RANKING ==========")
print("Primary   → Pass Rate (highest first)")
print("Tiebreak1 → Median Score (highest first)")
print("Tiebreak2 → Min Score (highest first — best worst case)")
print("Tiebreak3 → Enrollment Rate (lowest first — utterance ref preferred)\n")

summary_df["_pass_num"] = summary_df["Pass Rate"].apply(
    lambda x: int(x.split("/")[0])
)
summary_df["_enrollment_num"] = summary_df["Enrollment Rate"].apply(
    lambda x: int(x.split("/")[0])
)

ranking = summary_df.sort_values(
    by=[
        "_pass_num",        # Pass Rate — highest first
        "Median Score",     # Median — highest first
        "Min Score",        # Min — highest first
        "_enrollment_num",  # Enrollment count — lowest first
    ],
    ascending=[
        False,  # Pass Rate — higher is better
        False,  # Median Score — higher is better
        False,  # Min Score — higher is better
        True,   # Enrollment count — lower is better
    ]
)[[
    "Model", "Pass Rate", "Median Score", "Min Score", "Enrollment Rate"
]]

print(ranking.to_string(index=False))


========== FULL PER-SEGMENT RESULTS ==========
Model  Sample  Score Pass      Ref Type
   m1  YASH 2  0.965    ✅ UTTERANCE_REF
   m1 YASH_01  0.965    ✅ UTTERANCE_REF

========== MODEL COMPARISON SUMMARY ==========
Model  Segments  Mean Score  Median Score  Min Score Pass Rate Enrollment Rate
   m1         2       0.965         0.965      0.965       2/2             0/2

========== MODEL RANKING ==========
Primary   → Pass Rate (highest first)
Tiebreak1 → Median Score (highest first)
Tiebreak2 → Min Score (highest first — best worst case)
Tiebreak3 → Enrollment Rate (lowest first — utterance ref preferred)

Model Pass Rate  Median Score  Min Score Enrollment Rate
   m1       2/2         0.965      0.965             0/2


In [10]:
# final cell in each gate notebook
df.to_csv(os.path.join(BASE_DIR, "results.csv"), index=False)
print("✅ Results saved to results.csv")

✅ Results saved to results.csv
